# World Cup Sentiment Tracker

## Data Exploration **(Real Tweets)** 

---

### First Dataset:

Source type:  
Replies/comments under official football tweets  
(less likely to be biased and/or noisy since official posts are usually neutral/promotional, but the replies are fan reactions)

Accounts:
- @FIFAWorldCup
- national team accounts
- major player/team accounts if relevant

Tweet types:
- goal announcements
- full-time result posts
- VAR/penalty/red-card posts
- lineup posts
- highlight clips

Collected unit:  
Individual reply/comment, not the official post itself

----
### Strategy:

1. Pick one match/event
2. Find official posts about that event
3. Pull replies under those posts
4. Store replies with metadata
5. Run baseline sentiment
6. Manually inspect model failures

---

## Real Football Tweet Collection

Goal:
Collect replies from official football event posts and evaluate how well the baseline sentiment model performs on real football fan language.

Questions:
1. Can we successfully collect real football discussion?
2. Does the baseline sentiment model perform reasonably well?
3. What types of football language cause model failures?

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

from api.getxapi_client import (
    fetch_replies,
    normalize_replies
)
from data.preprocess import (
    annotate_replies,
    preprocess_reply_buckets,
    preprocess_replies,
    preprocessing_summary
)
from models.sentiment_baseline import (
    add_sentiment,
    load_sentiment_model
)

In [2]:
response = fetch_replies(tweet_id="2069468618476200189")

response.keys()

dict_keys(['tweetId', 'reply_count', 'has_more', 'next_cursor', 'replies'])

In [3]:
rows = normalize_replies(
    response=response,
    match="Portugal vs Spain",
    event="goal",
    source_account="FIFAWorldCup",
    team="Portugal"
)

raw_df = pd.DataFrame(rows)

raw_df.head()

,tweet_id,parent_tweet_id,url,timestamp,text,author_username,author_name,like_count,reply_count,view_count,match,team,player,event,source_account,source
0,2069547886975635584,2069468618476200189,https://x.com/NoCaptionMood/status/20695478869...,Tue Jun 23 22:27:21 +0000 2026,@FIFAWorldCup https://x.com/i/status/206952949...,NoCaptionMood,No Caption Mood,71,1,20243,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
1,2069547912825049173,2069468618476200189,https://x.com/grok/status/2069547912825049173,Tue Jun 23 22:27:27 +0000 2026,@NoCaptionMood @FIFAWorldCup Ask Grok is curre...,grok,Grok,111,1,11451,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
2,2044086411951808699,2069468618476200189,https://x.com/hyperagentapp/status/20440864119...,Tue Apr 14 16:12:32 +0000 2026,42 agents. 216 threads. One dashboard. Every a...,hyperagentapp,Hyperagent,10240,0,73896692,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
3,2069469280991637775,2069468618476200189,https://x.com/DPicwin/status/2069469280991637775,Tue Jun 23 17:15:00 +0000 2026,"@FIFAWorldCup No one—I repeat, no one can ever...",DPicwin,D. Picwin,445,17,18167,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
4,2069468946906886206,2069468618476200189,https://x.com/Shameless_925/status/20694689469...,Tue Jun 23 17:13:40 +0000 2026,@FIFAWorldCup Stupid record. Messi has 18 WC g...,Shameless_925,Amanda❤️🌸,45,38,19380,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi


In [4]:
raw_df.columns

Index(['tweet_id', 'parent_tweet_id', 'url', 'timestamp', 'text',
       'author_username', 'author_name', 'like_count', 'reply_count',
       'view_count', 'match', 'team', 'player', 'event', 'source_account',
       'source'],
      dtype='str')

In [5]:
raw_df.to_csv(
    "../data/raw/real_replies_sample.csv",
    index=False
)

In [6]:
raw_df.shape

(38, 16)

In [7]:
raw_df["text"].head(20)

0     @FIFAWorldCup https://x.com/i/status/206952949...
1     @NoCaptionMood @FIFAWorldCup Ask Grok is curre...
2     42 agents. 216 threads. One dashboard. Every a...
3     @FIFAWorldCup No one—I repeat, no one can ever...
4     @FIFAWorldCup Stupid record. Messi has 18 WC g...
5     @FIFAWorldCup He has won how many World cup tr...
6     @FIFAWorldCup Messi's fans right now🤣🤣🤣 https:...
7     @FIFAWorldCup Relax… it’s just against Uzbekis...
8     @FIFAWorldCup CRISTIANO RONALDO IS THE NAME!!\...
9     @FIFAWorldCup When Ronaldo scores\nThe whole w...
10    @FIFAWorldCup MESSI and MBAPPE right now: http...
11    @FIFAWorldCup How many total World Cup goals d...
12    @FIFAWorldCup @Unserious_Dc Our goat 🐐 is cook...
13    @FIFAWorldCup The only player to score in six ...
14    @FIFAWorldCup The older he gets, the less time...
15    Designed to delight, built to last. Get iPhone...
16    @FIFAWorldCup Well, goals against Uzbekistan d...
17                               @FIFAWorldCup T

In [8]:
reply_buckets = preprocess_reply_buckets(raw_df)

annotated_df = reply_buckets["annotated"]
analysis_df = reply_buckets["analysis"]
review_df = reply_buckets["review"]
removed_df = reply_buckets["removed"]

preprocessing_summary(annotated_df, analysis_df)

{'raw_rows': 38,
 'analysis_rows': 32,
 'removed_rows': 6,
 'filter_reasons': {'keep': 32, 'unusable_text': 3, 'spam': 3}}

In [9]:
pd.set_option("display.max_colwidth", None)

review_df[
    [
        "text",
        "clean_text",
        "parent_context_confidence",
        "contextual_terms",
        "context_relevance_boost",
        "needs_context_review",
        "relevance_score"
    ]
].head(20)

,text,clean_text,parent_context_confidence,contextual_terms,context_relevance_boost,needs_context_review,relevance_score
0,"@FIFAWorldCup No one—I repeat, no one can ever achieve what this man has achieved https://t.co/ZDXVNGg2aL","No one—I repeat, no one can ever achieve what this man has achieved",3,"[achieve, achieved, man, no one, this man]",2,True,2
1,@FIFAWorldCup Another record https://t.co/u32Bczokqt,Another record,3,"[another record, record]",2,True,2
2,@FIFAWorldCup Is there a woman who has done this then @grok,Is there a woman who has done this then,3,[],1,True,1
3,@FIFAWorldCup Another record set! https://t.co/oCaTMNXXjZ,Another record set!,3,"[another record, record]",2,True,2
4,@FIFAWorldCup 📹 credit:FIFA https://t.co/wHEFCcCvka,📹 credit:FIFA,3,[],1,True,1
5,@FIFAWorldCup the GREATEST🫶🏻♾️ https://t.co/Q8zz6ULbFB,the GREATEST🫶🏻♾️,3,[],1,True,1
6,@FIFAWorldCup HE’S FUCKING BACK BABYY😭 https://t.co/GAAOINIfiy,HE’S FUCKING BACK BABYY😭,3,"[back, he]",2,True,2
7,Build intelligent voice experiences without rebuilding your stack.,Build intelligent voice experiences without rebuilding your stack.,3,[],1,True,1


In [10]:
analysis_df[
    [
        "text",
        "clean_text",
        "matched_entities",
        "inferred_teams",
        "inferred_players",
        "inferred_managers",
        "entity_confidence",
        "parent_context_confidence",
        "contextual_terms",
        "context_relevance_boost",
        "needs_context_review",
        "relevance_score",
        "filter_reason"
    ]
].head(20)

,text,clean_text,matched_entities,inferred_teams,inferred_players,inferred_managers,entity_confidence,parent_context_confidence,contextual_terms,context_relevance_boost,needs_context_review,relevance_score,filter_reason
0,"42 agents. 216 threads. One dashboard. Every agent gets its own prompt, tools, skills, and budget. Deploy specialized agents across your company. From the team at Airtable.","42 agents. 216 threads. One dashboard. Every agent gets its own prompt, tools, skills, and budget. Deploy specialized agents across your company. From the team at Airtable.",[team],[],[],[],1,3,[],1,False,4,keep
1,"@FIFAWorldCup No one—I repeat, no one can ever achieve what this man has achieved https://t.co/ZDXVNGg2aL","No one—I repeat, no one can ever achieve what this man has achieved",[],[],[],[],0,3,"[achieve, achieved, man, no one, this man]",2,True,2,keep
2,@FIFAWorldCup Stupid record. Messi has 18 WC goals https://t.co/CxPOqFJSWD,Stupid record. Messi has 18 WC goals,[messi],[Argentina],[Lionel Messi],[],1,3,[record],2,False,4,keep
3,@FIFAWorldCup He has won how many World cup trophies? @grok 😂😂 https://t.co/r3tM4hV8DA,He has won how many World cup trophies? 😂😂,"[cup, world cup]",[],[],[],2,3,[he],2,False,8,keep
4,@FIFAWorldCup Messi's fans right now🤣🤣🤣 https://t.co/FcFd4StXrT,Messi's fans right now🤣🤣🤣,[messi],[Argentina],[Lionel Messi],[],1,3,[],1,False,3,keep
5,@FIFAWorldCup Relax… it’s just against Uzbekistan 🇺🇿 😂 https://t.co/mG8AG42zkb,Relax… it’s just against Uzbekistan 🇺🇿 😂,[uzbekistan],[Uzbekistan],[],[],1,3,[],1,False,3,keep
6,@FIFAWorldCup CRISTIANO RONALDO IS THE NAME!!\nhttps://t.co/qfXt8tBCY5,CRISTIANO RONALDO IS THE NAME!!,"[cristiano, cristiano ronaldo, ronaldo]",[Portugal],[Cristiano Ronaldo],[],3,3,[],1,False,7,keep
7,@FIFAWorldCup When Ronaldo scores\nThe whole world talks about Ronaldo \n\nWhen Messi scores\nThe whole world talk about Ronaldo \n\nThe Real GOAT.,When Ronaldo scores The whole world talks about Ronaldo When Messi scores The whole world talk about Ronaldo The Real GOAT.,"[goat, messi, ronaldo]","[Argentina, Portugal]","[Cristiano Ronaldo, Lionel Messi]",[],3,3,[],1,False,8,keep
8,@FIFAWorldCup MESSI and MBAPPE right now: https://t.co/cZwaDLTMvt,MESSI and MBAPPE right now:,"[mbappe, messi]","[Argentina, France]","[Kylian Mbappé, Lionel Messi]",[],2,3,[],1,False,5,keep
9,"@FIFAWorldCup How many total World Cup goals does he have? Surely more than Messi, right?","How many total World Cup goals does he have? Surely more than Messi, right?","[cup, messi, world cup]",[Argentina],[Lionel Messi],[],3,3,[he],2,False,10,keep


In [14]:
classifier = load_sentiment_model()

analysis_df = add_sentiment(
    analysis_df,
    text_column="clean_text",
    classifier=classifier
)

analysis_df[
    [
        "text",
        "clean_text",
        "sentiment_label",
        "sentiment_score",
        "relevance_score",
        "needs_context_review"
    ]
].head(20)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,text,clean_text,sentiment_label,sentiment_score,relevance_score,needs_context_review
0,"42 agents. 216 threads. One dashboard. Every agent gets its own prompt, tools, skills, and budget. Deploy specialized agents across your company. From the team at Airtable.","42 agents. 216 threads. One dashboard. Every agent gets its own prompt, tools, skills, and budget. Deploy specialized agents across your company. From the team at Airtable.",NEGATIVE,0.975363,4,False
1,"@FIFAWorldCup No one—I repeat, no one can ever achieve what this man has achieved https://t.co/ZDXVNGg2aL","No one—I repeat, no one can ever achieve what this man has achieved",NEGATIVE,0.997070,2,True
2,@FIFAWorldCup Stupid record. Messi has 18 WC goals https://t.co/CxPOqFJSWD,Stupid record. Messi has 18 WC goals,NEGATIVE,0.995983,4,False
3,@FIFAWorldCup He has won how many World cup trophies? @grok 😂😂 https://t.co/r3tM4hV8DA,He has won how many World cup trophies? 😂😂,POSITIVE,0.992692,8,False
4,@FIFAWorldCup Messi's fans right now🤣🤣🤣 https://t.co/FcFd4StXrT,Messi's fans right now🤣🤣🤣,POSITIVE,0.995415,3,False
5,@FIFAWorldCup Relax… it’s just against Uzbekistan 🇺🇿 😂 https://t.co/mG8AG42zkb,Relax… it’s just against Uzbekistan 🇺🇿 😂,POSITIVE,0.922164,3,False
6,@FIFAWorldCup CRISTIANO RONALDO IS THE NAME!!\nhttps://t.co/qfXt8tBCY5,CRISTIANO RONALDO IS THE NAME!!,POSITIVE,0.999237,7,False
7,@FIFAWorldCup When Ronaldo scores\nThe whole world talks about Ronaldo \n\nWhen Messi scores\nThe whole world talk about Ronaldo \n\nThe Real GOAT.,When Ronaldo scores The whole world talks about Ronaldo When Messi scores The whole world talk about Ronaldo The Real GOAT.,POSITIVE,0.997308,8,False
8,@FIFAWorldCup MESSI and MBAPPE right now: https://t.co/cZwaDLTMvt,MESSI and MBAPPE right now:,POSITIVE,0.988174,5,False
9,"@FIFAWorldCup How many total World Cup goals does he have? Surely more than Messi, right?","How many total World Cup goals does he have? Surely more than Messi, right?",POSITIVE,0.990058,10,False


In [15]:
sample = analysis_df.sample(
    min(25, len(analysis_df)),
    random_state=42
)

sample[
    [
        "text",
        "clean_text",
        "matched_entities",
        "inferred_teams",
        "inferred_players",
        "inferred_managers",
        "needs_context_review",
        "sentiment_label",
        "sentiment_score",
        "relevance_score"
    ]
]

,text,clean_text,matched_entities,inferred_teams,inferred_players,inferred_managers,needs_context_review,sentiment_label,sentiment_score,relevance_score
29,@FIFAWorldCup siuuuuuuuuuuu https://t.co/e8oZrTvduQ,siuuuuuuuuuuu,[siuuu],[Portugal],[Cristiano Ronaldo],[],False,NEGATIVE,0.984447,2
15,@FIFAWorldCup Another record https://t.co/u32Bczokqt,Another record,[],[],[],[],True,POSITIVE,0.990748,2
24,@FIFAWorldCup Thats Goat baby https://t.co/VwFYGq7kJh,Thats Goat baby,[goat],[],[],[],False,NEGATIVE,0.924199,4
17,"@FIFAWorldCup Porto, Portugal 🇵🇹 https://t.co/HCg4iH82yt","Porto, Portugal 🇵🇹",[portugal],[Portugal],[],[],False,POSITIVE,0.962682,5
8,@FIFAWorldCup MESSI and MBAPPE right now: https://t.co/cZwaDLTMvt,MESSI and MBAPPE right now:,"[mbappe, messi]","[Argentina, France]","[Kylian Mbappé, Lionel Messi]",[],False,POSITIVE,0.988174,5
9,"@FIFAWorldCup How many total World Cup goals does he have? Surely more than Messi, right?","How many total World Cup goals does he have? Surely more than Messi, right?","[cup, messi, world cup]",[Argentina],[Lionel Messi],[],False,POSITIVE,0.990058,10
30,@FIFAWorldCup @RohitvNiranjan I love him...hope he lifts the cup this time.🐐❤️,I love him...hope he lifts the cup this time.🐐❤️,[cup],[],[],[],False,POSITIVE,0.999834,5
25,@FIFAWorldCup Is he back to being the goat again?,Is he back to being the goat again?,[goat],[],[],[],False,NEGATIVE,0.996956,5
12,"@FIFAWorldCup The older he gets, the less time he needs to change a game.","The older he gets, the less time he needs to change a game.",[game],[],[],[],False,NEGATIVE,0.994009,5
0,"42 agents. 216 threads. One dashboard. Every agent gets its own prompt, tools, skills, and budget. Deploy specialized agents across your company. From the team at Airtable.","42 agents. 216 threads. One dashboard. Every agent gets its own prompt, tools, skills, and budget. Deploy specialized agents across your company. From the team at Airtable.",[team],[],[],[],False,NEGATIVE,0.975363,4


### Observations

Real reply data contains media-only, text-poor, spam-like, low-relevance, and context-dependent replies. The current preprocessing is intentionally recall-oriented: it keeps more potentially relevant football discussion, including vague replies that rely on parent tweet context, while accepting that some junk can still slip into `analysis_df`.

The working pipeline is now:

```text
Raw API replies
-> normalized rows
-> clean text
-> football entity detection
-> parent-context relevance boost
-> inferred team/player/manager metadata
-> data quality filtering
-> baseline sentiment labels and scores
```

`analysis_df` is the dataset sent to the baseline sentiment model. `review_df` is the subset of kept rows that rely mainly on parent context and should be manually inspected. `removed_df` keeps clear junk and unusable text for audit.

This loose filter is acceptable for the MVP because it prioritizes retaining real fan reactions over perfectly excluding every irrelevant reply. A future production version could add a second-stage relevance classifier or embedding similarity check to reduce false positives.

Filtering and enrichment are implemented in `src/data/preprocess.py` and `src/data/entities.py`, with reference data in `data/reference/football_entities.json`. Sentiment scoring is applied with the baseline DistilBERT model through `src/models/sentiment_baseline.py`.

This notebook concludes pulling and preprocessing data from the API, yet the issue of our baseline sentiment model's subpar accuracy still remains.